In [ ]:
import os

import cmocean
import h5py
import numpy as np
from matplotlib import pyplot as plt

# Visualize all channels for a single randomly drawn pre-training sample.
# Rows = channels, Columns = time steps.

N_TIMESTEPS = 5
SEED = None  # set to an int for reproducibility
TARGET_T = N_TIMESTEPS // 2

CHANNELS = [
    dict(name="LST",              norm_range=(180.0, 360.0),   cmap=cmocean.cm.matter_r, vmin=273.15, vmax=323.15),
    dict(name="Air temperature",  norm_range=(205.13, 311.35), cmap=cmocean.cm.matter_r, vmin=273.15, vmax=323.15),
    dict(name="LAI",              norm_range=(0.0, 14.447),    cmap=cmocean.cm.algae,    vmin=0,      vmax=4),
    dict(name="Surface moisture", norm_range=(0.007, 0.857),   cmap=cmocean.cm.dense,    vmin=0,      vmax=0.85),
]

_esdc_data_dir = os.environ.get(
    "ESDC_DATA_DIR",
    "../../../esdc/data/lst_no_syn_gaps",
)
h5_path = os.path.join(_esdc_data_dir, "dataset.h5")

In [ ]:
rng = np.random.default_rng(SEED)

with h5py.File(h5_path, "r") as f:
    n_total = f["X_train"].shape[0]
    idx = int(rng.integers(0, n_total))
    sample = f["X_train"][idx].astype(np.float32)  # (T, H, W, C=4)
    times = f["time_train"][idx]                   # (2,) — start/end of sequence

t_start = times[0].decode() if isinstance(times[0], bytes) else str(times[0])
t_end   = times[1].decode() if isinstance(times[1], bytes) else str(times[1])
date_range = f"{t_start[:10]} → {t_end[:10]}"

print(f"Selected sample idx={idx}  (from {n_total} total)")
print(f"Date range: {date_range}")
print(f"Sample shape: {sample.shape}")

In [ ]:
n_ch = len(CHANNELS)
n_t = N_TIMESTEPS

fig, axes = plt.subplots(
    n_ch, n_t, figsize=(3.5 * n_t, 3 * n_ch),
    squeeze=False,
)

for c, ch in enumerate(CHANNELS):
    lo, hi = ch["norm_range"]
    cmap = ch["cmap"].copy()
    # cmap.set_bad("lightgray")
    cmap.set_bad("red")

    for t in range(n_t):
        ax = axes[c][t]
        img = sample[t, :, :, c]                      # (H, W), normalized [0, 1]
        phys = img * (hi - lo) + lo                    # denormalize to physical units
        masked = np.ma.masked_invalid(phys)

        ax.imshow(masked, cmap=cmap, vmin=ch["vmin"], vmax=ch["vmax"],
                  interpolation="nearest")

        gap_pct = 100.0 * float(np.isnan(img).sum()) / img.size
        title = f"t={t}"
        if t == TARGET_T:
            title += " (target)"
        if gap_pct > 0:
            title += f"\ngaps={gap_pct:.1f}%"
        ax.set_title(title, fontsize=8)
        ax.axis("off")

        if t == 0:
            ax.set_ylabel(ch["name"], fontsize=10, rotation=0, labelpad=35, va="center")
            ax.yaxis.set_label_coords(-0.15, 0.5)
            ax.axis("on")
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(False)

fig.suptitle(f"All channels — sample {idx}  ({date_range})", fontsize=13, y=1.002)
plt.tight_layout()
plt.show()